# (Transformers) 디코딩 전략 실습: temperature / top_p / top_k (+ min_p 워퍼)

vLLM 실습에 들어가기 전에, `transformers.generate()`의 기본 디코딩 옵션들을 정리합니다.

- Greedy / Beam Search / Sampling
- Sampling에서 `temperature`, `top_p`, `top_k` 변화 관찰
- (보너스) `min_p`를 **logits warper**로 직접 구현해보기

> 강의 포인트: "샘플링 제어"가 결국 *logits → 확률 분포 → 토큰 선택* 과정의 후보군을 어떻게 자르느냐의 문제라는 감을 잡기


## 0) 설치

- GPU 환경이면 `torch` + `transformers`를 설치합니다.
- 사내 이미지에 이미 설치되어 있다면 스킵 가능합니다.


In [ ]:

# !pip -q install "torch" "transformers" "accelerate" "huggingface_hub" pandas matplotlib jsonschema

import os, re, json, time
import torch
import pandas as pd
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModelForCausalLM


/opt/conda/envs/sam/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1) (옵션) Hugging Face 로그인

Gemma/Llama 등 일부 모델은 라이선스/게이트가 걸려있을 수 있습니다.
그 경우 `HF_TOKEN` 환경변수를 설정한 뒤 로그인하세요.

```python
export HF_TOKEN=...
```


In [ ]:

from huggingface_hub import login

hf_token = os.getenv("HF_TOKEN", "")
if hf_token:
    login(token=hf_token)
    print("Logged in to Hugging Face.")
else:
    print("HF_TOKEN not found. (공개 모델이면 로그인 없이도 진행 가능)")


HF_TOKEN not found. (공개 모델이면 로그인 없이도 진행 가능)


## 2) 모델 로딩

강의에서는 작은 모델로 시작하는 걸 추천합니다.

- 예시) `Qwen/Qwen2.5-1.5B-Instruct`, `TinyLlama/TinyLlama-1.1B-Chat-v1.0` 등

환경에 맞게 `MODEL_ID`를 바꿔주세요.


In [ ]:

MODEL_ID = os.getenv("MODEL_ID", "Qwen/Qwen2.5-1.5B-Instruct")
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto" if device == "cuda" else None,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
)

model.eval()


device: cuda


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:01<00:00, 209.77it/s]


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

## 3) 생성 헬퍼 함수

- `do_sample=False` → greedy/beam 류
- `do_sample=True` → sampling(temperature/top_p/top_k 등 사용)

결과 비교가 쉽도록 DataFrame 형태로 정리합니다.


In [ ]:

from transformers import GenerationConfig

def generate_one(
    prompt: str,
    do_sample: bool = True,
    temperature: float = 1.0,
    top_p: float = 0.9,
    top_k: int = 0,
    num_beams: int = 1,
    max_new_tokens: int = 200,
    min_p=0,
) -> str:
    text_prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt=True,
        tokenize=False,
    )

    inputs = tokenizer(text_prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        out = model.generate(
            **inputs,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
            top_p=top_p if do_sample else None,
            top_k=top_k if do_sample else None,
            num_beams=num_beams,
            max_new_tokens=max_new_tokens,
            min_p=min_p,
        )

    generated_ids = out[0][input_len:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()



PROMPT = "오늘 저녁 뭐 먹지? 간단히 3가지 추천해줘."

# 스모크 테스트
print(generate_one(PROMPT, do_sample=True, temperature=0.7, top_p=0.9, top_k=0))


1. 김치찌개: 대표적인 한국 요리로 맛있고 부드럽다.
2. 삼겹살: 건강하면서도 고소한 음식으로 선택하기 좋은 옵션이다.
3. 미역국: 향긋하고 단백질이 많은 해산물 요리로 좋다.


## 4) 기본 비교: Greedy vs Sampling

- Greedy는 항상 가장 확률 높은 토큰만 고르므로 “안정적”이지만 다양성이 낮습니다.
- Sampling은 다양성을 얻는 대신 high temperature에서 출력이 흔들릴 수 있습니다.


In [ ]:

rows = []
rows.append({"mode": "greedy", "text": generate_one(PROMPT, do_sample=False, max_new_tokens=160)})
rows.append({"mode": "sample_T0.7_top_p0.9", "text": generate_one(PROMPT, do_sample=True, temperature=0.7, top_p=0.9, max_new_tokens=160)})
rows.append({"mode": "sample_T1.6_top_p0.9", "text": generate_one(PROMPT, do_sample=True, temperature=1.6, top_p=0.9, max_new_tokens=160)})

pd.DataFrame(rows)


,mode,text
0,greedy,물론이죠! 오늘 저녁은 다음과 같은 3가지 요리들을 추천드립니다:\n\n1. 김치찌...
1,sample_T0.7_top_p0.9,물론이죠! 다양한 맛을 즐길 수 있도록 간단하게 3가지를 추천드리겠습니다:\n\n1...
2,sample_T1.6_top_p0.9,Rebellion Kalipus Mulherkrụするとreceipt공 {?阿里に出す...


## 5) (보너스) min_p를 logits warper로 직접 구현

`min_p`는 보통 “최대 확률 토큰에 비해 너무 확률이 낮은 토큰”을 후보에서 제거하는 방식으로 구현할 수 있습니다.



### high temperature에서 min_p 효과 관찰

- `temperature=1.6` 고정
- `top_p=0.9` 고정
- `min_p` 유무만 비교


In [ ]:


rows = []
rows.append({"mode": "T1.6_top_p0.9", "text": generate_one(PROMPT, do_sample=True, temperature=1.6, top_p=0.9,)})
rows.append({"mode": "T1.6_top_p0.9_min_p0.08", "text": generate_one(PROMPT, do_sample=True, temperature=1.6, top_p=0.9, min_p=0.08)})
pd.DataFrame(rows)


,mode,text
0,T1.6_top_p0.9,"알 Stateside Hüh Anyway에 모리도(rawֵdeǘㄅ金山 мед비此种""..."
1,T1.6_top_p0.9_min_p0.08,1. 김치찌개 (다양한 종류의 김치와 맛있는 채소를 활용해서 만들어진 한식 대표菜品...


## 6) 미니 과제

1) temperature를 1.8~2.0으로 올려서 출력이 얼마나 흔들리는지 관찰  
2) `min_p`를 0.03~0.15로 스윕하며 “말도 안 되는 출력”이 줄어드는지 확인  
3) 너무 큰 min_p는 다양성/창의성을 과하게 억제할 수 있음을 토론

다음 노트북에서는 같은 아이디어를 **vLLM**에서 더 빠르게(배치/서빙) 실험합니다.
